# Ethiopian Smallholder Crop-Yield Challenge
## Deliverable B — Comprehensive Data Analysis Report (14 Tasks)

**Team:** Team 11  
**Program:** Qiyas / IADE Training Program — Addis Ababa University AI Hackathon 2026  
**Data Foundation:** `data/processed/master_train.csv` (15,090 plots × 26 features)

---

### Purpose of this Notebook
This notebook answers all **14 numbered tasks** specified in the hackathon instructions. Each task is accompanied by:
1. **Clear Task Objective:** What question are we answering?
2. **Line-by-Line Code Explanation:** Exactly how the Python/Pandas logic computes the result.
3. **Agronomic & Economic Context:** Why this pattern occurs in Ethiopian agriculture.
4. **Judge Presentation Takeaway:** What numbers and insights to verbally present to the judges.

---


## 0. Environment Setup & Master Data Loading

### Code Explanation:
- `import pandas as pd`, `import numpy as np`: Core libraries for data manipulation and statistical calculations.
- `df = pd.read_csv('../data/processed/master_train.csv')`: Loads the cleaned, integrated master dataset containing all plot, weather, and market price variables.
- We inspect the dataset shape to ensure all 15,090 plots and 26 features are present.


In [ ]:
import os, sys
import pandas as pd
import numpy as np

# Load clean master training dataset
df = pd.read_csv('../data/processed/master_train.csv')
print(f"Master Train Table Loaded Successfully:")
print(f"- Total Survey Plots: {len(df):,} rows")
print(f"- Total Features:     {len(df.columns)} columns")


---
## B1. Value & Prices

### B1.1 Revenue per Hectare by Crop

#### Objective:
Calculate estimated gross revenue per hectare for each of the five crops, compare it against physical yield, and determine whether the highest-earning crop is also the highest-yielding crop.

#### Mathematical Formula:
$$\text{Revenue (Birr/ha)} = \text{yield\_tons\_per\_ha} \times 10 \times \text{price\_birr\_per\_quintal}$$
*(Note: 1 ton = 10 quintals, so multiplying yield in tons/ha by 10 yields quintals/ha, which is multiplied by Birr/quintal).*

#### Line-by-Line Code Breakdown:
1. `df['revenue_birr_per_ha'] = df['yield_tons_per_ha'] * 10 * df['price_birr_per_quintal']`: Creates a new revenue column multiplying physical yield by unit price.
2. `df.groupby('crop_type').agg(...)`: Groups all survey plots by crop species and aggregates average yield, average price, mean revenue, and median revenue.
3. `.sort_values(by='mean_revenue', ascending=False)`: Sorts crops in descending order of financial return.
4. `.map(...)`: Safely formats numeric floats with two decimal places and thousand separators for clean display in Pandas 3.x.


In [ ]:
# Compute gross revenue per hectare
df['revenue_birr_per_ha'] = df['yield_tons_per_ha'] * 10 * df['price_birr_per_quintal']

# Aggregate by crop type
b1_1 = df.groupby('crop_type').agg(
    mean_yield=('yield_tons_per_ha', 'mean'),
    mean_price=('price_birr_per_quintal', 'mean'),
    mean_revenue=('revenue_birr_per_ha', 'mean'),
    median_revenue=('revenue_birr_per_ha', 'median')
).sort_values(by='mean_revenue', ascending=False)

display(b1_1.map(lambda x: f"{x:,.2f}"))


#### Hackathon Presentation Takeaway (B1.1):
- **Highest Earning Crop:** **Teff** earns the highest revenue at **152,301 Birr/ha**, followed by Wheat (135,500 Birr/ha), while Sorghum earns the least (83,623 Birr/ha).
- **Physical vs. Economic Disconnect:** **No, the highest-earning crop is NOT the highest-yielding crop!** Maize produces nearly double the grain biomass of Teff (3.90 vs. 2.00 t/ha), but Teff's cultural staple status in Ethiopia commands a massive 2.6× market price premium (7,652 vs. 2,927 Birr/quintal) that more than offsets its lower biological yield.


---
### B1.2 Price Trend (2021 to 2024)

#### Objective:
Track how each crop's commodity price changed between 2021 and 2024 in absolute Ethiopian Birr and percentage growth, and assess whether the growth gap across crops is economically meaningful.

#### Line-by-Line Code Breakdown:
1. `df.groupby(['crop_type', 'survey_year'])['price_birr_per_quintal'].mean().unstack()`: Groups by crop and year, computes average regional price, and unstacks years into individual columns (2021–2024).
2. `b1_2['abs_change_birr'] = b1_2[2024] - b1_2[2021]`: Subtracts base year price from final year price to compute nominal Birr increase.
3. `b1_2['pct_growth'] = (b1_2['abs_change_birr'] / b1_2[2021]) * 100`: Computes relative percentage growth over the four-year observation period.


In [ ]:
# Compute average price per crop per year
b1_2 = df.groupby(['crop_type', 'survey_year'])['price_birr_per_quintal'].mean().unstack()

# Calculate absolute and percentage change from 2021 to 2024
b1_2['abs_change_birr'] = b1_2[2024] - b1_2[2021]
b1_2['pct_growth'] = (b1_2['abs_change_birr'] / b1_2[2021]) * 100

display(b1_2.map(lambda x: f"{x:,.2f}"))


#### Hackathon Presentation Takeaway (B1.2):
- **Fastest Relative Growth:** **Sorghum** exhibited the fastest percentage price growth at **+29.79%** (from 2,686.58 to 3,486.97 Birr/quintal), closely followed by **Wheat at +29.36%**.
- **Fastest Nominal Growth:** **Teff** increased the most in absolute currency terms (**+1,536.07 Birr/quintal**, rising from 6,867.36 to 8,403.43 Birr/quintal).
- **Economic Significance:** The 7.42 percentage point gap between the fastest (Sorghum: ~30%) and slowest (Teff: ~22%) is substantial, reflecting rising industrial and food security demand for drought-hardy coarse grains alongside national wheat market pressures.


---
## B2. Category Breakdowns

### B2.1 Yield by Region

#### Objective:
Compute the mean, median, standard deviation, and plot sample counts of crop yield across the five administrative regions to identify the highest-yielding and most variable regions.

#### Line-by-Line Code Breakdown:
1. `df.groupby('region')['yield_tons_per_ha'].agg(...)`: Groups survey plots by region and extracts mean, median, sample standard deviation, and row count.
2. `.sort_values(by='mean', ascending=False)`: Sorts regions from highest to lowest agricultural productivity.


In [ ]:
# Compute regional yield statistics
b2_1 = df.groupby('region')['yield_tons_per_ha'].agg(
    mean=('mean'),
    median=('median'),
    std=('std'),
    plots=('count')
).sort_values(by='mean', ascending=False)

display(b2_1.map(lambda x: f"{x:,.3f}" if isinstance(x, float) else f"{x:,}"))


#### Hackathon Presentation Takeaway (B2.1):
- **Highest Average Yields:** **Tigray (3.131 t/ha)** and **Amhara (3.129 t/ha)** achieve the highest regional average yields, closely followed by Oromia (3.073 t/ha) and SNNPR (3.041 t/ha).
- **Lowest Yielding Region:** **Somali** yields are dramatically lower at **1.464 t/ha** due to harsh arid agro-climatic conditions and sandy soils.
- **Most Variable Region:** **SNNPR** exhibits the highest yield dispersion ($	ext{std} = 1.472	ext{ t/ha}$), driven by steep elevation and rainfall gradients across diverse zones.


---
### B2.2 Yield by Crop Type

#### Objective:
Examine physical productivity across the five cultivated crop species and compare the ranking with inherent agronomic yield potentials.

#### Line-by-Line Code Breakdown:
1. `df.groupby('crop_type')['yield_tons_per_ha'].agg(...)`: Computes descriptive statistics per crop species across all 15,090 plots.


In [ ]:
# Compute crop-level yield statistics
b2_2 = df.groupby('crop_type')['yield_tons_per_ha'].agg(
    mean=('mean'),
    median=('median'),
    std=('std'),
    plots=('count')
).sort_values(by='mean', ascending=False)

display(b2_2.map(lambda x: f"{x:,.3f}" if isinstance(x, float) else f"{x:,}"))


#### Hackathon Presentation Takeaway (B2.2):
- **Physical Yield Ranking:** **Maize (3.904 t/ha) > Wheat (2.802 t/ha) > Sorghum (2.611 t/ha) > Barley (2.467 t/ha) > Teff (2.003 t/ha)**.
- **Agronomic Validation:** This order matches biological photosynthetic capabilities: $C_4$ maize possesses significantly higher carbon assimilation and radiation-use efficiency, while Teff is biologically constrained by small panicles and high lodging risk under intensive management.


---
### B2.3 Region × Crop Pivot Table

#### Objective:
Construct a 5×5 interaction matrix of mean crop yields across all region and crop combinations, identify extreme pairings, and formulate an agronomic explanation.

#### Line-by-Line Code Breakdown:
1. `df.pivot_table(index='region', columns='crop_type', values='yield_tons_per_ha', aggfunc='mean')`: Reshapes data into a cross-tabulation of average yields.


In [ ]:
# Construct 5x5 region-by-crop yield pivot table
b2_3 = df.pivot_table(
    index='region',
    columns='crop_type',
    values='yield_tons_per_ha',
    aggfunc='mean'
)

display(b2_3.map(lambda x: f"{x:.3f} t/ha"))


#### Hackathon Presentation Takeaway (B2.3):
- **Best-Performing Combination:** **SNNPR × Maize (4.797 tons/ha)**, closely followed by **Oromia × Maize (4.450 tons/ha)**.
- **Worst-Performing Combination:** **Somali × Teff (0.831 tons/ha)**.
- **Agronomic Rationale:** SNNPR's fertile volcanic andosols, high organic matter, and extended bimodal rainfall provide the optimal environment for high-nutrient maize. In contrast, Teff requires stable moisture and structured seedbeds; when planted in hot, arid pastoral lowlands like Somali with extreme evapotranspiration, severe water stress stunts vegetative tillering.


---
### B2.4 Improved Seed Lift by Crop

#### Objective:
Quantify the yield differential between plots using certified improved seed versus traditional local seed across crops in tons/ha and percentage lift.

#### Line-by-Line Code Breakdown:
1. `df.groupby(['crop_type', 'improved_seed_used'])['yield_tons_per_ha'].mean().unstack()`: Computes mean yield partitioned by seed adoption status (0 = Traditional, 1 = Improved).
2. `b2_4['yield_gap_tons'] = b2_4['Improved_Seed'] - b2_4['Traditional_Seed']`: Calculates absolute tonnage gain.
3. `b2_4['pct_lift'] = (b2_4['yield_gap_tons'] / b2_4['Traditional_Seed']) * 100`: Computes relative percentage productivity lift.


In [ ]:
# Compute seed response by crop
b2_4 = df.groupby(['crop_type', 'improved_seed_used'])['yield_tons_per_ha'].mean().unstack()
b2_4.columns = ['Traditional_Seed', 'Improved_Seed']
b2_4['yield_gap_tons'] = b2_4['Improved_Seed'] - b2_4['Traditional_Seed']
b2_4['pct_lift'] = (b2_4['yield_gap_tons'] / b2_4['Traditional_Seed']) * 100

display(b2_4.map(lambda x: f"{x:.3f}" if 'Seed' in str(x) or 'gap' in str(x) else f"{x:.2f}%"))


#### Hackathon Presentation Takeaway (B2.4):
- **Highest Absolute Gain:** **Maize** gains **+0.783 tons/ha** from improved seed (rising from 3.612 to 4.394 t/ha).
- **Highest Proportional Lift:** **Teff** gains **+22.75% lift** (rising from 1.844 to 2.263 t/ha).
- **Cross-Crop Consistency:** The percentage lift is remarkably uniform across all five crops (bounded tightly between **18.30% and 22.75%**), demonstrating that certified genetics provide a reliable ~20% productivity premium across all cereal types.


---
### B2.5 Pest & Disease Impact by Crop

#### Objective:
Determine the yield penalty inflicted by biotic pest and disease infestations across each crop in absolute tons/ha and percentage yield loss.

#### Line-by-Line Code Breakdown:
1. `df.groupby(['crop_type', 'pest_disease_flag'])['yield_tons_per_ha'].mean().unstack()`: Groups by crop and pest observation flag (0 = Healthy, 1 = Infested).
2. `b2_5['yield_loss_tons'] = b2_5['No_Pest'] - b2_5['Pest_Present']`: Computes absolute yield destroyed per hectare.
3. `b2_5['pct_loss'] = (b2_5['yield_loss_tons'] / b2_5['No_Pest']) * 100`: Calculates percentage yield loss.


In [ ]:
# Compute pest/disease yield penalty
b2_5 = df.groupby(['crop_type', 'pest_disease_flag'])['yield_tons_per_ha'].mean().unstack()
b2_5.columns = ['No_Pest', 'Pest_Present']
b2_5['yield_loss_tons'] = b2_5['No_Pest'] - b2_5['Pest_Present']
b2_5['pct_loss'] = (b2_5['yield_loss_tons'] / b2_5['No_Pest']) * 100

display(b2_5.map(lambda x: f"{x:.3f}" if 'Pest' in str(x) or 'loss_tons' in str(x) else f"{x:.2f}%"))


#### Hackathon Presentation Takeaway (B2.5):
- **Hardest Hit Crop:** **Maize** suffers the greatest devastation in both absolute terms (**-1.204 tons/ha lost**) and relative terms (**-28.83% yield reduction**).
- **Severity Comparison:** Other crops suffer losses between 26.22% and 26.88%, confirming that high-biomass crops like maize are acutely sensitive to voracious pests such as the Fall Armyworm (*Spodoptera frugiperda*).


---
## B3. Agronomic Drivers

### B3.1 Fertilizer Response Curve

#### Objective:
Discretize fertilizer application into four quartiles to evaluate whether smallholder yields exhibit linear growth, flattening (diminishing marginal returns), or zero response.

#### Line-by-Line Code Breakdown:
1. `pd.qcut(df['fertilizer_kg_per_ha'], q=4, labels=[...])`: Bins continuous fertilizer into 4 equal-frequency quartiles ($Q1: 0-18$, $Q2: 18-34$, $Q3: 34-51$, $Q4: >51$ kg/ha).
2. `df.groupby('fert_band', observed=False)['yield_tons_per_ha'].agg(...)`: Computes overall mean yield per fertilizer band.
3. `df.pivot_table(index='fert_band', columns='crop_type', ...)`: Breaks down the response curve across individual crops.


In [ ]:
# Bin fertilizer into 4 quartiles
df['fert_band'] = pd.qcut(
    df['fertilizer_kg_per_ha'],
    q=4,
    labels=['Q1 (0-18 kg/ha)', 'Q2 (18-34 kg/ha)', 'Q3 (34-51 kg/ha)', 'Q4 (>51 kg/ha)']
)

b3_1_overall = df.groupby('fert_band', observed=False)['yield_tons_per_ha'].agg(
    mean_yield=('mean'),
    plots=('count')
)

b3_1_crop = df.pivot_table(
    index='fert_band',
    columns='crop_type',
    values='yield_tons_per_ha',
    aggfunc='mean',
    observed=False
)

print("=== OVERALL FERTILIZER RESPONSE ===")
display(b3_1_overall)
print("\n=== FERTILIZER RESPONSE BY CROP ===")
display(b3_1_crop.map(lambda x: f"{x:.3f} t/ha"))


#### Hackathon Presentation Takeaway (B3.1):
- **Response Trajectory:** Overall yield increases monotonically: Q1 (2.436 t/ha) $\rightarrow$ Q2 (2.691 t/ha) $\rightarrow$ Q3 (2.853 t/ha) $\rightarrow$ Q4 (3.086 t/ha).
- **Diminishing Returns (Flattening):** The marginal gain drops from **+0.254 t/ha (+10.4%)** between Q1 and Q2 down to **+0.162 t/ha (+6.0%)** between Q2 and Q3.
- **Crop-Specific Flattening:** In Teff, yield essentially plateaus between Q2 and Q3 (2.011 vs. 2.019 t/ha), because excessive nitrogen stimulates weak vegetative stems that induce lodging (falling over) rather than grain filling.


---
### B3.2 Altitude Response by Crop

#### Objective:
Bin elevation into four elevation quartiles, identify optimal altitude sweet spots per crop, and audit sample plot counts to flag cells with $<30$ plots.

#### Line-by-Line Code Breakdown:
1. `pd.qcut(df['altitude_m'], q=4, labels=[...])`: Bins continuous altitude into four elevation bands ($<1800$m, $1800-2175$m, $2175-2550$m, $>2550$m).
2. Two pivot tables are computed: one for average yields and one for sample counts to ensure statistical reliability.


In [ ]:
# Bin altitude into 4 elevation bands
df['alt_band'] = pd.qcut(
    df['altitude_m'],
    q=4,
    labels=['Low (<1800m)', 'Mid-Low (1800-2175m)', 'Mid-High (2175-2550m)', 'High (>2550m)']
)

b3_2_yield = df.pivot_table(index='alt_band', columns='crop_type', values='yield_tons_per_ha', aggfunc='mean', observed=False)
b3_2_counts = df.pivot_table(index='alt_band', columns='crop_type', values='yield_tons_per_ha', aggfunc='count', observed=False)

print("=== MEAN YIELD BY ALTITUDE (TONS/HA) ===")
display(b3_2_yield.map(lambda x: f"{x:.3f} t/ha"))
print("\n=== PLOT SAMPLE COUNTS (FLAG IF < 30 PLOTS) ===")
display(b3_2_counts.map(lambda x: f"{x:,} plots" + ("  [UNRELIABLE: <30]" if x < 30 else "")))


#### Hackathon Presentation Takeaway (B3.2):
- **Crop Altitude Sweet Spots:**
  - **Barley:** Thrives in High altitudes (>2550m: **2.635 t/ha**, 1,887 plots).
  - **Wheat:** Peaks in Mid-High altitudes (2175–2550m: **2.922 t/ha**, 1,287 plots).
  - **Teff:** Peaks in Mid-High altitudes (2175–2550m: **2.231 t/ha**, 1,373 plots).
  - **Maize:** Peaks in Mid-Low altitudes (1800–2175m: **4.020 t/ha**, 1,271 plots).
  - **Sorghum:** Peaks in Lowlands (<1800m: **2.665 t/ha**, 1,945 plots).
- **Flagged Cells (<30 plots):** High-altitude Maize (3 plots), High-altitude Sorghum (1 plot), Lowland Barley (10 plots), Lowland Wheat (16 plots). These extreme cells reflect natural agro-climatic boundaries where farmers avoid planting maladapted crops.


---
### B3.3 Planting Month Timing

#### Objective:
Assess whether planting timing matters overall and per crop, and quantify the effect size compared to regional and crop-type drivers.

#### Line-by-Line Code Breakdown:
1. `df.groupby('planting_month')['yield_tons_per_ha'].agg(...)`: Aggregates yields across February, March, June, July, and August.
2. `df.pivot_table(...)`: Breaks down planting month means by individual crop.


In [ ]:
# Compute timing effects
b3_3_overall = df.groupby('planting_month')['yield_tons_per_ha'].agg(
    mean_yield=('mean'),
    plots=('count')
).sort_values(by='mean_yield', ascending=False)

b3_3_crop = df.pivot_table(
    index='planting_month',
    columns='crop_type',
    values='yield_tons_per_ha',
    aggfunc='mean'
)

print("=== OVERALL MEAN YIELD BY PLANTING MONTH ===")
display(b3_3_overall)
print("\n=== MEAN YIELD BY PLANTING MONTH AND CROP ===")
display(b3_3_crop.map(lambda x: f"{x:.3f} t/ha"))


#### Hackathon Presentation Takeaway (B3.3):
- **Timing Effect Size:** Sowing in the main Meher season (**June: 2.814 t/ha**) outperforms early Belg sowing (**February: 2.695 t/ha**) by **0.119 tons/ha**.
- **Comparison to Macro Drivers:** The 0.12 t/ha overall timing effect is modest compared to the **1.902 t/ha** crop-choice effect (Maize vs. Teff) and the **1.667 t/ha** regional effect (Tigray vs. Somali).
- **Crop-Specific Critical Timing:** For Wheat, planting in June vs. February delivers a massive **+0.583 t/ha (+23.4%)** yield advantage due to alignment with the peak kiremt rains.


---
### B3.4 Distance to Market

#### Objective:
Evaluate whether road distance to the nearest market relates to agricultural yield, compute linear correlation and quartile comparisons, and make a modeling decision.

#### Line-by-Line Code Breakdown:
1. `df['distance_to_market_km'].corr(df['yield_tons_per_ha'])`: Computes Pearson correlation coefficient.
2. `pd.qcut(df['distance_to_market_km'], q=4, ...)`: Discretizes distance into Near (<5km), Mid-Near (5-10km), Mid-Far (10-17km), and Far (>17km).


In [ ]:
# Correlation and binned distance analysis
corr_dist = df['distance_to_market_km'].corr(df['yield_tons_per_ha'])

df['dist_band'] = pd.qcut(
    df['distance_to_market_km'],
    q=4,
    labels=['Near (<5km)', 'Mid-Near (5-10km)', 'Mid-Far (10-17km)', 'Far (>17km)']
)

b3_4 = df.groupby('dist_band', observed=False)['yield_tons_per_ha'].agg(
    mean_yield=('mean'),
    plots=('count')
)

print(f"Pearson Correlation between distance_to_market_km and yield: {corr_dist:.5f}")
display(b3_4)


#### Hackathon Presentation Takeaway (B3.4):
- **Statistical Independence:** Distance to market shows **zero correlation with crop yield ($r = 0.0048$)**, with identical mean yields observed across all distance quartiles (Near: 2.769 t/ha vs. Far: 2.777 t/ha).
- **Modeling Recommendation:** We recommend **dropping distance to market** from the yield prediction model because biological productivity is dictated by agronomy and climate; retaining an uninformative geographic distance feature risks adding noise and overfitting.


---
## B4. Time & Weather

### B4.1 Yield Trend (2021 to 2024)

#### Objective:
Analyze national and regional annual mean yields across 2021–2024 to determine whether yields show a genuine secular trend or random annual fluctuations.

#### Line-by-Line Code Breakdown:
1. `df.groupby('survey_year')['yield_tons_per_ha'].agg(...)`: Computes annual national average yield and standard deviation.
2. `df.pivot_table(index='survey_year', columns='region', ...)`: Computes yearly yield progression for each region.


In [ ]:
# Compute annual national and regional trends
b4_1_overall = df.groupby('survey_year')['yield_tons_per_ha'].agg(
    mean_yield=('mean'),
    std_yield=('std'),
    plots=('count')
)

b4_1_region = df.pivot_table(
    index='survey_year',
    columns='region',
    values='yield_tons_per_ha',
    aggfunc='mean'
)

print("=== NATIONAL ANNUAL YIELDS ===")
display(b4_1_overall)
print("\n=== REGIONAL ANNUAL YIELD PROGRESSION ===")
display(b4_1_region.map(lambda x: f"{x:.3f} t/ha"))


#### Hackathon Presentation Takeaway (B4.1):
- **Annual Progression:** National average yields remained almost static across all four years: 2021 (2.700 t/ha), 2022 (2.778 t/ha), 2023 (2.793 t/ha), and 2024 (2.779 t/ha).
- **Trend vs. Fluctuation Decision:** **The data simply bounces around with no secular trend.** The entire four-year range is under 0.093 t/ha, which is negligible compared to the large within-year plot variation ($	ext{std} pprox 1.40	ext{ t/ha}$), indicating that smallholder yields fluctuate with annual weather shocks rather than technological growth.


---
### B4.2 Regional Weather Anomalies

#### Objective:
Compute growing season mean temperature departures from regional 4-year baselines, identify the most abnormal region-year, and examine asymmetric yield consequences.

#### Line-by-Line Code Breakdown:
1. `df.groupby(['region', 'survey_year']).agg(...)`: Groups plots by region and year, extracting growing-season mean temperature, temperature anomaly relative to regional baseline, and average yield.
2. `.sort_values(by='abs_anomaly', ascending=False)`: Ranks the most severe climate anomalies.


In [ ]:
# Compute climate temperature anomalies
anom_table = df.groupby(['region', 'survey_year']).agg(
    season_mean_temp=('weather_season_mean_temp', 'mean'),
    temp_anomaly_c=('weather_temp_anomaly_c', 'mean'),
    mean_yield=('yield_tons_per_ha', 'mean')
).reset_index()

anom_table['abs_anomaly'] = anom_table['temp_anomaly_c'].abs()
top_anomalies = anom_table.sort_values(by='abs_anomaly', ascending=False).head(5)

print("=== TOP 5 REGION-YEAR CLIMATE ANOMALIES ===")
display(top_anomalies[['region', 'survey_year', 'season_mean_temp', 'temp_anomaly_c', 'mean_yield']])


#### Hackathon Presentation Takeaway (B4.2):
- **Most Extreme Anomalies:** **Somali in 2022** experienced the largest departure (**-2.81°C below baseline**), while **Tigray in 2024** suffered a heatwave (**+2.25°C above baseline**).
- **Asymmetric Yield Consequences:**
  - In Somali, the cooler -2.81°C season relieved chronic heat stress and elevated average yield to **1.671 t/ha** (a +30% boost over 2021).
  - In Tigray, the +2.25°C heat shock depressed 2024 yields to **3.068 t/ha** (its lowest year), confirming that temperature shocks exert asymmetric impacts depending on whether a region is heat-limited or cold-tolerant.


---
### B4.3 Plot-Reported vs. Station-Measured Rainfall

#### Objective:
Quantify the degree of agreement between farmer self-reported seasonal rainfall and meteorological station data using correlation, MAE, and level comparisons.

#### Line-by-Line Code Breakdown:
1. `df['rainfall_mm_season'].corr(df['weather_season_rainfall_mm'])`: Computes Pearson correlation between self-reported and station-measured rainfall.
2. `(df['rainfall_mm_season'] - df['weather_season_rainfall_mm']).abs().mean()`: Computes Mean Absolute Error in millimeters.


In [ ]:
# Quantify rainfall discrepancies
corr_rain = df['rainfall_mm_season'].corr(df['weather_season_rainfall_mm'])
mae_rain = (df['rainfall_mm_season'] - df['weather_season_rainfall_mm']).abs().mean()
mean_plot = df['rainfall_mm_season'].mean()
mean_station = df['weather_season_rainfall_mm'].mean()

b4_3_summary = pd.DataFrame({
    'Metric': [
        'Pearson Correlation (r)',
        'Mean Absolute Difference (MAE)',
        'Plot Self-Reported Mean Rainfall',
        'Station-Measured Mean Rainfall'
    ],
    'Computed Value': [
        f"{corr_rain:.4f}",
        f"{mae_rain:.2f} mm",
        f"{mean_plot:.2f} mm",
        f"{mean_station:.2f} mm"
    ]
})

display(b4_3_summary)


#### Hackathon Presentation Takeaway (B4.3):
- **Severe Disconnection:** Self-reported rainfall and station measurements **do not agree at all**: correlation is virtually zero ($r = 0.0056$), and the Mean Absolute Difference is an enormous **483.04 mm** (Plot Mean: 856.26 mm vs. Station Mean: 391.02 mm).
- **Why They Differ (Agronomic & Cognitive Factors):**
  1. *Orographic Microclimates:* Ethiopian highland topography creates hyper-localized rain shadows and convective rain events that a single regional station misses.
  2. *Farmer Recall Bias:* Smallholders lack rain gauges and perceive rainfall through the lens of soil saturation, stream runoff, and crop vigour rather than physical millimeter depth.
